In [ ]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============= STEP 1: LOAD FILES =============
print("Loading files...")

# Load all input files
mapping_table = pd.read_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/mapping table.xlsx')
all_lists = pd.read_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/All lists.xlsx')
all_loans = pd.read_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/All loans.xlsx')
sheet1_template = pd.read_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/Sheet 1.xlsx')

print(f"✓ Loaded {len(all_lists)} inventory items")
print(f"✓ Loaded {len(all_loans)} loan records")
print(f"✓ Loaded mapping table with {len(mapping_table)} mappings")

# ============= STEP 2: MAP INVENTORY (ALL_LISTS) =============
print("\nMapping inventory items...")

# Add new columns to all_lists
all_lists['Primary Category'] = None
all_lists['Product Name'] = None
all_lists['Unknown Category'] = None

# Map each item in all_lists
for i, row in all_lists.iterrows():
    item_type = row['Item Type']
    
    # Find this item type in mapping table
    mapping = mapping_table[mapping_table['Type'] == item_type]
    
    if not mapping.empty:
        # Get the first matching row
        m = mapping.iloc[0]
        
        # If has Primary Category and Product Name, use them
        if pd.notna(m['Primary Category']) and pd.notna(m['Product Name']):
            all_lists.at[i, 'Primary Category'] = m['Primary Category']
            all_lists.at[i, 'Product Name'] = m['Product Name']
        # If only has Primary Category
        elif pd.notna(m['Primary Category']):
            all_lists.at[i, 'Primary Category'] = m['Primary Category']
        # If no Primary Category but has Unknown
        elif pd.notna(m['Unknown']):
            all_lists.at[i, 'Unknown Category'] = m['Unknown']

print(f"✓ Mapped {len(all_lists)} inventory items")

# Save mapped inventory
all_lists.to_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/All_lists_Mapped.xlsx', index=False)

# ============= STEP 3: CREATE NAME-TO-CATEGORY DICTIONARY =============
print("\nCreating name mapping for loans...")

# Create dictionary from item names to categories
name_mapping = {}
for i, row in all_lists.iterrows():
    name = row['Name']
    if pd.notna(name):
        name_mapping[name] = {
            'Primary Category': row['Primary Category'],
            'Product Name': row['Product Name'],
            'Unknown Category': row['Unknown Category']
        }

# ============= STEP 4: MAP LOANS (ALL_LOANS) =============
print("Mapping loan items...")

# Add new columns to all_loans
all_loans['Primary Category'] = None
all_loans['Product Name'] = None
all_loans['Unknown Category'] = None

# Map each loan by item name
for i, row in all_loans.iterrows():
    item_name = row['Item Name']
    
    # Look up this item name in our mapping
    if item_name in name_mapping:
        mapping = name_mapping[item_name]
        all_loans.at[i, 'Primary Category'] = mapping['Primary Category']
        all_loans.at[i, 'Product Name'] = mapping['Product Name']
        all_loans.at[i, 'Unknown Category'] = mapping['Unknown Category']

print(f"✓ Mapped {len(all_loans)} loan records")

# Save mapped loans
all_loans.to_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/All_loans_Mapped.xlsx', index=False)

# ============= STEP 5: CREATE SHEET 1 (INVENTORY/REPAIR) =============
print("\nCreating Sheet 1 (Repair)...")

sheet1 = sheet1_template.copy()
sheet1['Quantity'] = 0
sheet1['Scenario'] = 'Repair'

# Count inventory items for each category-product pair
for i, row in sheet1.iterrows():
    category = row['Primary Category']
    product = row['Product Name']
    
    if pd.notna(category) and pd.notna(product):
        # Count matching items in inventory
        count = len(all_lists[
            (all_lists['Primary Category'] == category) & 
            (all_lists['Product Name'] == product)
        ])
        sheet1.at[i, 'Quantity'] = count

# Add unmapped items from inventory
unmapped_items = []

# ========== CHANGE 1: NOW USING 'Name' INSTEAD OF 'Item Type' FOR REPAIR ITEMS ==========
# Find items with category but no product
cat_only = all_lists[(all_lists['Primary Category'].notna()) & 
                     (all_lists['Product Name'].isna())]
if not cat_only.empty:
    # CHANGED: Now grouping by 'Name' instead of 'Item Type'
    groups = cat_only.groupby(['Primary Category', 'Name']).size()
    for (cat, item_name), count in groups.items():  # CHANGED: item_name instead of item_type
        unmapped_items.append({'Category': cat, 'Item': item_name, 'Count': count})

# Find items with Unknown category
unknown = all_lists[all_lists['Unknown Category'].notna()]
if not unknown.empty:
    # CHANGED: Now grouping by 'Name' instead of 'Item Type'
    groups = unknown.groupby(['Unknown Category', 'Name']).size()
    for (cat, item_name), count in groups.items():  # CHANGED: item_name instead of item_type
        unmapped_items.append({'Category': cat, 'Item': item_name, 'Count': count})

# ========== CHANGE 2: USING 'Unmapped Item' COLUMN INSTEAD OF 'Unmapped Type' ==========
# Add unmapped items to sheet1
if 'Unmapped Item' not in sheet1.columns:  # CHANGED: 'Unmapped Item' instead of 'Unmapped Type'
    sheet1['Unmapped Item'] = None

for item in unmapped_items:
    # Find where to insert this row (after last row of this category)
    cat_rows = sheet1[sheet1['Primary Category'] == item['Category']]
    if not cat_rows.empty:
        insert_at = cat_rows.index[-1] + 1
    else:
        insert_at = len(sheet1)
    
    # Create new row
    new_row = pd.DataFrame({
        'Primary Category': [item['Category']],
        'Product Name': [None],
        'Quantity': [item['Count']],
        'Weight': [None],
        'Scenario': ['Repair'],
        'Unmapped Item': [item['Item']]  # CHANGED: 'Unmapped Item' instead of 'Unmapped Type'
    })
    
    # Insert it
    sheet1 = pd.concat([sheet1.iloc[:insert_at], new_row, 
                        sheet1.iloc[insert_at:]]).reset_index(drop=True)

# Save Sheet 1
sheet1.to_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/Sheet_1_Repair.xlsx', index=False)
print(f"✓ Created Sheet 1 with {len(sheet1)} rows")

# ============= STEP 6: CREATE SHEET 2 (LOANS/RENTAL) =============
print("Creating Sheet 2 (Rental)...")

sheet2 = sheet1_template.copy()
sheet2['Quantity'] = 0
sheet2['Scenario'] = 'Rental'

# Count loan items for each category-product pair
for i, row in sheet2.iterrows():
    category = row['Primary Category']
    product = row['Product Name']
    
    if pd.notna(category) and pd.notna(product):
        # Count matching items in loans
        count = len(all_loans[
            (all_loans['Primary Category'] == category) & 
            (all_loans['Product Name'] == product)
        ])
        sheet2.at[i, 'Quantity'] = count

# Add unmapped loan items
unmapped_loans = []

# Find loans with category but no product
cat_only = all_loans[(all_loans['Primary Category'].notna()) & 
                     (all_loans['Product Name'].isna())]
if not cat_only.empty:
    groups = cat_only.groupby(['Primary Category', 'Item Name']).size()
    for (cat, item_name), count in groups.items():
        unmapped_loans.append({'Category': cat, 'Item': item_name, 'Count': count})

# Find loans with Unknown category
unknown = all_loans[all_loans['Unknown Category'].notna()]
if not unknown.empty:
    groups = unknown.groupby(['Unknown Category', 'Item Name']).size()
    for (cat, item_name), count in groups.items():
        unmapped_loans.append({'Category': cat, 'Item': item_name, 'Count': count})

# Add unmapped items to sheet2
if 'Unmapped Item' not in sheet2.columns:
    sheet2['Unmapped Item'] = None

for item in unmapped_loans:
    # Find where to insert this row
    cat_rows = sheet2[sheet2['Primary Category'] == item['Category']]
    if not cat_rows.empty:
        insert_at = cat_rows.index[-1] + 1
    else:
        insert_at = len(sheet2)
    
    # Create new row
    new_row = pd.DataFrame({
        'Primary Category': [item['Category']],
        'Product Name': [None],
        'Quantity': [item['Count']],
        'Weight': [None],
        'Scenario': ['Rental'],
        'Unmapped Item': [item['Item']]
    })
    
    # Insert it
    sheet2 = pd.concat([sheet2.iloc[:insert_at], new_row, 
                        sheet2.iloc[insert_at:]]).reset_index(drop=True)

# Save Sheet 2
sheet2.to_excel('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/Sheet_2_Rental.xlsx', index=False)
print(f"✓ Created Sheet 2 with {len(sheet2)} rows")

# ============= STEP 7: CREATE COMBINED SHEET 3 =============
print("\nCreating combined Sheet 3...")

# Get all unique categories
all_categories = sorted(set(
    list(sheet1['Primary Category'].dropna()) + 
    list(sheet2['Primary Category'].dropna())
))

# Combine sheets by category
combined = []
for category in all_categories:
    # First add all Repair rows for this category
    repair_rows = sheet1[sheet1['Primary Category'] == category]
    if not repair_rows.empty:
        combined.append(repair_rows)
    
    # Then add all Rental rows for this category
    rental_rows = sheet2[sheet2['Primary Category'] == category]
    if not rental_rows.empty:
        combined.append(rental_rows)

# Combine into single DataFrame
sheet3 = pd.concat(combined, ignore_index=True)

# Create summary
summary = []
for category in all_categories:
    repair_count = sheet3[(sheet3['Primary Category'] == category) & 
                          (sheet3['Scenario'] == 'Repair')]['Quantity'].sum()
    rental_count = sheet3[(sheet3['Primary Category'] == category) & 
                          (sheet3['Scenario'] == 'Rental')]['Quantity'].sum()
    summary.append({
        'Category': category,
        'Repair Items': int(repair_count),
        'Rental Items': int(rental_count),
        'Total': int(repair_count + rental_count)
    })

summary_df = pd.DataFrame(summary)

# Save Sheet 3 with summary
with pd.ExcelWriter('/Users/username/Documents/Seattle Reconomy files/Try 2/Code Files/Sheet_3_Combined.xlsx') as writer:
    sheet3.to_excel(writer, sheet_name='Combined Inventory', index=False)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"✓ Created Sheet 3 with {len(sheet3)} rows")

# ============= DONE! =============
print("\n" + "="*50)
print("SUCCESS! All files created:")
print("="*50)
print("1. All_lists_Mapped.xlsx - Inventory with categories")
print("2. All_loans_Mapped.xlsx - Loans with categories")
print("3. Sheet_1_Repair.xlsx - Inventory counts (Repair)")
print("4. Sheet_2_Rental.xlsx - Loan counts (Rental)")
print("5. Sheet_3_Combined.xlsx - Combined final sheet")
print("\nTotal items processed:")
print(f"  - Inventory: {len(all_lists):,} items")
print(f"  - Loans: {len(all_loans):,} records")
print(f"  - Combined rows: {len(sheet3)} rows")
print(f"  - Categories: {len(all_categories)} categories")

Loading files...
✓ Loaded 7438 inventory items
✓ Loaded 54937 loan records
✓ Loaded mapping table with 1774 mappings

Mapping inventory items...
✓ Mapped 7438 inventory items

Creating name mapping for loans...
Mapping loan items...
✓ Mapped 54937 loan records

Creating Sheet 1 (Repair)...
✓ Created Sheet 1 with 461 rows
Creating Sheet 2 (Rental)...
✓ Created Sheet 2 with 410 rows

Creating combined Sheet 3...
✓ Created Sheet 3 with 871 rows

SUCCESS! All files created:
1. All_lists_Mapped.xlsx - Inventory with categories
2. All_loans_Mapped.xlsx - Loans with categories
3. Sheet_1_Repair.xlsx - Inventory counts (Repair)
4. Sheet_2_Rental.xlsx - Loan counts (Rental)
5. Sheet_3_Combined.xlsx - Combined final sheet

Total items processed:
  - Inventory: 7,438 items
  - Loans: 54,937 records
  - Combined rows: 871 rows
  - Categories: 19 categories


In [8]:
import pandas as pd
pd.ExcelFile('/Users/shreyalokray/Documents/Seattle Reconomy files/Try 2/Code Files/All lists.xlsx').sheet_names


['Export']